# Precision at fixed candidate budgets, ranked by `TM_CCOEFF_NORMED` (K = 10, 20, 30, 50)

**Question.** Replicates `precision_at_k_budgets_14roi/precision_at_k_budgets_14roi.ipynb`
exactly, with one change: the template-matching method is `cv2.TM_CCOEFF_NORMED` instead of
`cv2.TM_CCOEFF`, used for both the response map and the ranking (D1 in `DECISIONS.md`). D1
rejected `TM_CCOEFF_NORMED` on a `read_50` comparison -- `TM_CCOEFF` won 49/49 decision-grade
cells, median ratio 4.32x -- but flagged that comparison as owed a re-derivation at the newer
`recall@K`/precision@K metric family, on the match score alone. That re-derivation is what this
notebook is, and Table C below states its verdict directly against the reference notebook's own
output. As in the reference notebook, `recall@K` is not reported here (see the closing summary
for where the recall-family columns end up instead).

**Scope, fixed for this run:**
- 14 ROIs, `images/extra_valid` (2 per tumour domain x 7 domains).
- 1 seed per ROI, `seed_index = 0` -- identical RNG stream to the reference notebook, so the
  same click is drawn on each ROI.
- Today's decided defaults (`DECISIONS.md`) that are held fixed: no `tissue_mask` (D2),
  `hematoxylin_od` unclipped (D3), ranked by `tm_score` (D5), NMS radius = match radius =
  7.5 um (D7, enforced via `invariants.check_nms_radius`). **D1 is the one axis deliberately not
  held fixed** -- `METHOD = cv2.TM_CCOEFF_NORMED` below, in place of the reference notebook's
  `cv2.TM_CCOEFF`. D3 (unclipped `hematoxylin_od`) is held fixed here simply because it is the
  decided default, independent of this method swap -- not because of an invariance argument.
  DECISIONS.md's own D1+D3 text says `TM_CCOEFF_NORMED` "cannot tell" whether the channel was
  rescaled, since it is invariant to `I -> aI + b`; that is exact only for a true global affine
  map, and the retired `hematoxylin` channel is not one -- it is affine on the bulk but clips
  (saturates) at the 0.5th/99.5th percentile tails, precisely on the darkest, densest chromatin
  pixels a mitotic-figure template is built from (D3's own point). A clipped template's internal
  contrast structure differs from the unclipped one, so even `TM_CCOEFF_NORMED`'s score against
  it would differ. Worth stating precisely rather than repeating the shorthand -- it does not
  change the decision to hold D3 fixed here regardless.
- No z-threshold sweep. Candidates are extracted once per ROI at a permissive deep floor
  (`z = -1.5`, the repo's standing "near-unfiltered" constant, computed from this run's own
  `template_match.robust_stats` -- the mitigation D1 itself requires for any non-default
  method), NMS'd at 7.5 um, ranked by score, and then `compare.evaluate_arms`'s own budget loop
  truncates to K = 10/20/30/50. Each ROI's own z at rank 10/20/30/50 is reported as a
  diagnostic column, not applied as a filter.
- **No pool-reproduction gate.** `precision_at_k_budgets_14roi_chromatin/`'s Gate 0 checks that
  its candidate pool exactly reproduces the TM_CCOEFF reference run's, because that notebook
  only changes the ranking step *after* an identical search and NMS. This notebook changes the
  matching method itself, so the response map, the deep-floor pool, its robust-z scale and the
  resulting ranking are all expected to differ from `results/precision_at_k_14roi_per_roi.csv`
  -- that divergence is the entire subject of this comparison, not a bug to gate against.
- **One extra check this notebook needs that the reference notebook did not.** The reference
  notebook's `TM_CCOEFF` can never produce a NaN response (see `template_match.fused_response`'s
  own docstring); `TM_CCOEFF_NORMED` can, on a zero-variance window, and that NaN is converted to
  a large negative finite sentinel before `robust_stats` samples the map. A finite sentinel is
  not caught by `robust_stats`'s `isfinite` filter, so a fourth check below counts and asserts
  zero sentinel-contaminated pixels in the sampled, `valid`-masked population, on every ROI.</cell id="3340958e">


In [1]:
import gc
import time
import sys

import cv2
import numpy as np
import pandas as pd
from skimage.measure import label, regionprops

sys.path.insert(0, '..')
from midog_utils import channels as ch
from midog_utils import dataset as ds
from midog_utils import evaluate as ev
from midog_utils import find_and_suppress as fs
from midog_utils import seed_selection as ss
from midog_utils import template_match as tm
from midog_utils import compare as cp
from midog_utils import invariants as inv
from midog_utils.nms import nms_by_distance

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 220)

NB_T0 = time.time()

# ---------------------------------------------------------------------------------------
# Configuration. Identical to `precision_at_k_budgets_14roi.ipynb` except METHOD: D2/D3/D5/D7
# per DECISIONS.md are held fixed, D1 is deliberately varied (TM_CCOEFF_NORMED in place of
# TM_CCOEFF). No z-sweep -- one deep-floor extraction per ROI, then evaluate_arms's own budget
# loop does the K = 10/20/30/50 truncation.
# ---------------------------------------------------------------------------------------
IMAGES_DIR = '../images/extra_valid'
SEED_INDEX = 0                       # one seed per ROI

CHANNEL = 'hematoxylin_od'           # D3 -- unclipped optical density
METHOD = cv2.TM_CCOEFF_NORMED         # deliberate D1 deviation -- normalized, contrast-blind
PEAK_MIN_DISTANCE = 7
SELF_HIT_RADIUS = 5.0
DEEP_FLOOR_Z = -1.5                  # near-unfiltered extraction floor (repo convention)
MAX_PEAKS = 2_000_000

NMS_RADIUS_UM = ev.MIDOG_RADIUS_UM   # 7.5 um -- D7: NMS radius == match radius
MATCH_RADIUS_UM = ev.MIDOG_RADIUS_UM

CFG = fs.FSConfig(channel=CHANNEL, base_size=tm.BASE_SIZE, scales=(1.0,),
                  n_angles=1, flips=(False,), peak_min_distance=PEAK_MIN_DISTANCE,
                  self_hit_radius=SELF_HIT_RADIUS)
BORDER = CFG.patch_size // 2         # 36
OTSU_WINDOW = tm.BASE_SIZE           # 51

BUDGETS = (10, 20, 30, 50)           # the fixed candidate-list lengths under test

OUT_PER_ROI = '../results/precision_at_k_14roi_prodseed_normed_per_roi.csv'
OUT_BY_DOMAIN = '../results/precision_at_k_14roi_prodseed_normed_by_domain.csv'
OUT_RAW = '../results/precision_at_k_14roi_prodseed_normed_raw.csv'
OUT_VERIF = '../results/precision_at_k_14roi_prodseed_normed_verification.csv'

print(f'budgets {BUDGETS} x 14 ROIs x 1 click (seed_index={SEED_INDEX})')
print(f'METHOD = TM_CCOEFF_NORMED (deliberate D1 deviation)')
print(f'NMS radius = match radius = {NMS_RADIUS_UM} um (D7)')

budgets (10, 20, 30, 50) x 14 ROIs x 1 click (seed_index=0)
METHOD = TM_CCOEFF_NORMED (deliberate D1 deviation)
NMS radius = match radius = 7.5 um (D7)


## The pipeline

Identical to the notebook this copies, with **one change: how the seed is refined**.

The ancestors of this notebook (`recall_workload_ledger.py` -> `find_and_suppress_high_threshold_precision.ipynb`
-> `precision_at_k_budgets_14roi.ipynb`) carried an inline `largest_cc_box`, which takes the
largest Otsu component in the click's 51 px crop **whether or not the click lands inside it** --
the largest-connected-component fallback `seed_selection.tighten_box_otsu` refuses by design
(`BBOX_TUNING.md` calls it *"a weak heuristic ... in a crowded crop it can grab an unrelated,
larger structure"*). It also kept the template centred on the raw click, correcting only the
size.

This notebook uses `seed_selection.tightened_template_box` (`DECISIONS.md` **D8**, 2026-09-09)
instead, which changes two things at once:

1. **The containment gate is enforced.** `center_tolerance=0`: the click's own pixel must be
   foreground of the accepted component, or the seed is refused (`None`) and
   `draw_seed_with_retry` redraws on the same RNG stream. The sanity gates behind it
   (`min_area=50`, `max_area_frac=0.85`, `min_solidity=0.5`) are unchanged -- they are
   `tighten_box_otsu`'s own defaults and the same values the inline helper passed.
2. **The template is recentred on the accepted component's own bbox centre**, not the click.
   D8's measurement on these 14 ROIs: `tighten_box_otsu` accepts 896/993 (90.2%) of unanimous
   border-filtered mitotic annotations, and among those the click-to-centre offset runs a
   median 3.10 px, 95th percentile 8.14 px, max 13.87 px -- against templates of roughly
   19-51 px, so not a small correction.

**The three consequences D8 says the caller owns, and how they are handled here:**

* **Self-hit removal moves with the centre** (D8 item 1). `suppress()` and the seed-annulus
  diagnostic are both referenced to `tpl_xy`, the recentred point, not to the click.
* **Border safety moves with the centre** (D8 item 2). `border_filter` runs on the click as
  before, then `_check` additionally verifies the full rotation-safe patch is readable at the
  recentred point, refusing the seed if not.
* **Ground truth does not move** (D8 item 3). `(cx, cy)` stays the click for `gt_eval`
  exclusion and for every MIDOG match-radius computation. Only where the template is built
  from changes; what counts as finding a mitotic figure does not.

Everything downstream of the seed -- channel, method, augmentation, deep floor, NMS radius,
ranking, budgets, tables -- is byte-identical to the notebook this copies.

In [2]:
# D8 (`DECISIONS.md`, 2026-09-09): seed sizing AND centring come from
# `seed_selection.tightened_template_box`, so the inline `_odd_local` / `largest_cc_box`
# pair this notebook's ancestors carried is gone. The two rules are not equivalent:
# `largest_cc_box` took the largest Otsu component in the click's crop whether or not the
# click landed in it -- the largest-connected-component fallback `tighten_box_otsu`
# refuses by design. `tightened_template_box` keeps that containment gate exactly as it
# stands (center_tolerance=0: the click's own pixel must be foreground of the accepted
# component, else the seed is refused and redrawn) and adds the component's own bbox
# centre. Its sanity gates are `tighten_box_otsu`'s defaults -- min_area=50,
# max_area_frac=0.85, min_solidity=0.5 -- the same values the inline helper used.


def draw_seed_with_retry(pool, rng, check_fn):
    '''Draw a row via `rng.integers`; on failure drop it and redraw on the same stream.'''
    working, retries = pool.copy(), 0
    while len(working) > 0:
        idx = int(rng.integers(len(working)))
        row = working.iloc[idx]
        result = check_fn(row)
        if result is not None:
            return row, result, retries
        working = working.drop(working.index[idx])
        retries += 1
    raise ValueError('seed pool exhausted -- no candidate passed check_fn')


def suppress(centers, scores, radius, ref_xy):
    '''NMS at `radius`, then drop the template's own self-correlation. Returns (centers, scores).

    `ref_xy` is the point the template was built from -- under D8 the accepted Otsu
    component's own bbox centre, not the raw click. The self-correlation peak lands where the
    template is centred, so self-hit removal (and the seed-annulus diagnostic below) must both
    be referenced there: D8, "what it costs", item 1.

    SELF_HIT_RADIUS is the repo default 5.0 px (`FSConfig.self_hit_radius`).
    `find_and_suppress.py` keeps this filter deliberately tight rather than using the match
    radius: the minimum spacing between two real MIDOG++ annotations anywhere in the dataset
    is 26.2 px, below the ~30 px match radius, so a match-radius filter could delete a
    legitimate detection of a neighbouring ground-truth object.
    '''
    keep = nms_by_distance(centers, scores, radius)
    c, s = centers[keep], scores[keep]
    if len(c):
        ok = np.hypot(c[:, 0] - ref_xy[0], c[:, 1] - ref_xy[1]) > SELF_HIT_RADIUS
        c, s = c[ok], s[ok]
    return c, s


def roi_files(images_dir=IMAGES_DIR):
    import os
    return sorted(f for f in os.listdir(images_dir) if f.endswith('.tiff'))


print(f'{len(roi_files())} ROIs on disk in {IMAGES_DIR}/')

14 ROIs on disk in ../images/extra_valid/


## The run

Per ROI: one `matchTemplate` pass (the expensive step, ~15 s), one extraction at the deep
floor, one NMS at 7.5 um. No re-run per budget -- `compare.evaluate_arms` computes
`tp_at_budget` / `budget_delivered` for every K in `BUDGETS` from that single ranked list.

In [3]:
def run_roi(fn, image_id, domain, anns):
    t0 = time.time()
    path = f'{IMAGES_DIR}/{fn}'
    rgb = ds.load_roi(path)
    mpp = ds.roi_mpp(path)
    roi_shape = rgb.shape
    match_radius = ev.radius_px(mpp, MATCH_RADIUS_UM)
    nms_radius = ev.radius_px(mpp, NMS_RADIUS_UM)
    hem = ch.to_channel(rgb, CHANNEL)
    gray_inv = ch.to_gray_inverted(rgb)
    H, W = hem.shape[:2]
    del rgb
    gc.collect()

    # --- the click -------------------------------------------------------------------
    gt = ds.image_annotations(anns, fn)
    seed_pool, flagged = ss.agreement_pool(gt[gt['category_id'] == ds.MITOTIC])
    seed_pool = ss.border_filter(seed_pool, BORDER, roi_shape)
    rng = np.random.default_rng([SEED_INDEX, image_id])

    def _check(row):
        # D8: `tightened_template_box` applies `tighten_box_otsu`'s containment gate --
        # the click's own pixel (center_tolerance=0) must land inside the accepted
        # component, or the seed is refused (None) and `draw_seed_with_retry` redraws on
        # the same RNG stream. It returns (base_size, center_x, center_y), the centre in
        # full-image coordinates.
        r = ss.tightened_template_box(gray_inv, float(row['cx']), float(row['cy']),
                                      otsu_window=OTSU_WINDOW)
        if r is None:
            return None
        # D8 "what it costs", item 2: border safety moves with the centre. `border_filter`'s
        # margin (BORDER) was sized for a click-centred read, and the recentred point can sit
        # further toward the ROI edge, so re-check that the full rotation-safe patch is
        # actually readable there instead of assuming the click's own check still covers it.
        if tm.read_padded_patch(hem, r[1], r[2], CFG.patch_size) is None:
            return None
        return r

    seed, seed_tpl, n_retries = draw_seed_with_retry(seed_pool, rng, _check)
    base_size, tpl_cx, tpl_cy = seed_tpl
    tpl_xy = (float(tpl_cx), float(tpl_cy))           # D8: where the template is built from
    seed_xy = (float(seed['cx']), float(seed['cy']))  # D8 item 3: ground truth stays the click
    tpl_offset = float(np.hypot(tpl_xy[0] - seed_xy[0], tpl_xy[1] - seed_xy[1]))
    seed_ann_id = int(seed['ann_id'])
    gt_eval = gt[gt['ann_id'] != seed_ann_id].reset_index(drop=True)
    n_gt = int((gt_eval['category_id'] == ds.MITOTIC).sum())
    del gray_inv
    gc.collect()

    # --- one match, one deep-floor extraction, one NMS -- no z-sweep --------------------
    patch = tm.read_padded_patch(hem, *tpl_xy, CFG.patch_size)
    templates, _ = tm.build_augmentations(patch, base_size, CFG.scales, CFG.n_angles, CFG.flips)
    PAD = max((t.shape[0] - 1) // 2 for t in templates)
    hem_p = cv2.copyMakeBorder(hem, PAD, PAD, PAD, PAD, borderType=cv2.BORDER_REPLICATE)
    fused_p, _, valid_p = tm.fused_response(hem_p, templates, CFG.scale_normalize, method=METHOD)
    fused = fused_p[PAD:PAD + H, PAD:PAD + W]
    valid = valid_p[PAD:PAD + H, PAD:PAD + W]
    assert bool(valid.all()), f'{fn}: padding left part of the ROI unreachable (PAD={PAD})'
    del hem_p, fused_p, valid_p, hem
    gc.collect()

    med, mad = tm.robust_stats(fused, valid)
    cut = med + DEEP_FLOOR_Z * mad
    # Sentinel guard: a zero-variance window is NaN under TM_CCOEFF_NORMED (impossible under
    # the reference notebook's TM_CCOEFF), and fused_response converts that NaN to the module's
    # own sentinel, template_match._FLOOR. robust_stats filters its sample with `isfinite`,
    # which does not catch a finite sentinel, so this counts it directly -- against
    # template_match._SENTINEL_CUT, the same named cutoff `_robust_z` already uses internally
    # for exactly this test, rather than a freshly invented threshold.
    n_sentinel_in_valid = int((fused[valid] < tm._SENTINEL_CUT).sum())
    centers, scores = tm.extract_peaks(fused, valid, PEAK_MIN_DISTANCE, cut, MAX_PEAKS)
    n_peaks = len(centers)
    assert n_peaks < MAX_PEAKS, f'{fn}: MAX_PEAKS is binding, raise it'
    c, s = suppress(centers, scores, nms_radius, tpl_xy)
    assert bool(np.all(np.diff(s) <= 0)), f'{fn}: post-NMS pool is not score-descending'
    pool = pd.DataFrame({'cx': c[:, 0], 'cy': c[:, 1], 'score': s})

    # Seed annulus: at NMS radius == match radius, NMS itself empties this before self-hit
    # removal ever runs (the self-correlation is the map's global maximum, kept first, and
    # suppresses everything within one NMS radius) -- so this must count zero here.
    d_seed = np.hypot(pool['cx'] - tpl_xy[0], pool['cy'] - tpl_xy[1])
    n_near_seed = int((d_seed <= match_radius).sum())

    arm = cp.Arm('tm_score_deep_floor', (lambda d=pool: d), rank_key='score',
                 seeded=True, z=DEEP_FLOOR_Z, z_dependent=True,
                 nms_radius=nms_radius, caps=(MAX_PEAKS,))
    ctx = dict(file_name=fn, tumor_type=domain, image_id=image_id,
               seed_ann_id=seed_ann_id, base_size=base_size,
               map_median=round(float(med), 5), mad_scale=round(float(mad), 5), mpp=mpp)
    checks = []
    out = cp.evaluate_arms([arm], gt_eval, match_radius, roi_shape=(H, W), mpp=mpp,
                            budgets=BUDGETS, context=ctx, checks=checks)

    z_at_rank = {}
    for k in BUDGETS:
        z_at_rank[k] = (float(pool['score'].iloc[k - 1] - med) / mad) if len(pool) >= k else np.nan
    out['z_at_rank'] = out['budget'].map(z_at_rank)

    checks.append(dict(check='seed_annulus_empty', label=fn, n_near_seed=n_near_seed,
                       match_radius_px=round(match_radius, 3), passed=bool(n_near_seed == 0)))
    checks.append(dict(check='no_sentinel_in_valid_sample', label=fn,
                       n_sentinel_in_valid=n_sentinel_in_valid, passed=bool(n_sentinel_in_valid == 0)))

    meta = dict(file_name=fn, tumor_type=domain, image_id=image_id, seed_ann_id=seed_ann_id,
                n_retries=n_retries, contested_seed_tier=bool(flagged), base_size=base_size,
                tpl_cx=tpl_xy[0], tpl_cy=tpl_xy[1], tpl_offset_px=round(tpl_offset, 3),
                mpp=mpp, roi_h=int(H), roi_w=int(W), pad_px=PAD,
                match_radius_px=match_radius, nms_radius_px=nms_radius,
                map_median=float(med), mad_scale=float(mad), n_gt_mitotic=n_gt,
                n_detections=len(pool), n_near_seed_annulus=n_near_seed,
                n_sentinel_in_valid=n_sentinel_in_valid,
                t_total_s=round(time.time() - t0, 1))
    del fused, valid, templates, patch, centers, scores, c, s
    gc.collect()
    print(f"[{fn}] {domain:32s} base={base_size:2d} n_detections={len(pool):6d} "
          f"n_gt={n_gt:3d} [{meta['t_total_s']:.0f}s]", flush=True)
    return out, pd.DataFrame(checks), meta

In [4]:
images, annotations = ds.load_annotations('../databases/MIDOG++.json')
ds.check_invariants(annotations)
meta_ix = images.set_index('file_name')[['image_id', 'tumor_type']]

files = roi_files()
missing = sorted(set(files) - set(meta_ix.index))
assert not missing, f'.tiff on disk absent from the annotation DB: {missing}'
assert len(files) == 14, f'expected 14 ROIs in {IMAGES_DIR}/, found {len(files)}'

t_run = time.time()
out_frames, check_frames, roi_meta = [], [], []
for fn in files:
    image_id = int(meta_ix.loc[fn, 'image_id'])
    domain = meta_ix.loc[fn, 'tumor_type']
    out, checks, m = run_roi(fn, image_id, domain, annotations)
    out_frames.append(out)
    check_frames.append(checks)
    roi_meta.append(m)
    gc.collect()

RAW = pd.concat(out_frames, ignore_index=True)
VERIF = pd.concat(check_frames, ignore_index=True)
ROI = pd.DataFrame(roi_meta).set_index('file_name')

print(f'\n{len(files)} ROIs x {len(BUDGETS)} budgets = {len(RAW)} rows in {time.time() - t_run:.0f}s')
ROI[['tumor_type', 'seed_ann_id', 'base_size', 'tpl_offset_px', 'n_gt_mitotic', 'n_detections',
     'match_radius_px', 'nms_radius_px', 'map_median', 'mad_scale', 'n_sentinel_in_valid',
     't_total_s']].round(3)

[013.tiff] human breast cancer              base=31 n_detections= 17863 n_gt= 17 [8s]
[094.tiff] human breast cancer              base=25 n_detections= 18731 n_gt= 81 [5s]
[201.tiff] canine lung cancer               base=51 n_detections= 15916 n_gt= 17 [5s]
[233.tiff] canine lung cancer               base=25 n_detections= 17549 n_gt= 17 [4s]
[245.tiff] canine lymphosarcoma             base=47 n_detections= 17726 n_gt= 89 [4s]
[246.tiff] canine lymphosarcoma             base=41 n_detections= 18223 n_gt=115 [4s]
[300.tiff] canine cutaneous mast cell tumor base=45 n_detections= 18111 n_gt=180 [5s]
[301.tiff] canine cutaneous mast cell tumor base=41 n_detections= 17843 n_gt=217 [4s]
[402.tiff] human neuroendocrine tumor       base=29 n_detections= 17731 n_gt=104 [5s]
[403.tiff] human neuroendocrine tumor       base=51 n_detections= 16270 n_gt= 52 [5s]
[459.tiff] canine soft tissue sarcoma       base=33 n_detections= 18087 n_gt=130 [4s]
[460.tiff] canine soft tissue sarcoma       base=47 n_

,tumor_type,seed_ann_id,base_size,tpl_offset_px,n_gt_mitotic,n_detections,match_radius_px,nms_radius_px,map_median,mad_scale,n_sentinel_in_valid,t_total_s
file_name,,,,,,,,,,,,
013.tiff,human breast cancer,254,31,4.272,17,17863,33.139,33.139,-0.014,0.205,0,7.7
094.tiff,human breast cancer,2512,25,2.062,81,18731,32.630,32.630,-0.025,0.226,0,5.2
201.tiff,canine lung cancer,4457,51,0.500,17,15916,30.222,30.222,-0.009,0.170,0,4.6
233.tiff,canine lung cancer,5761,25,1.581,17,17549,30.222,30.222,-0.025,0.255,0,4.3
245.tiff,canine lymphosarcoma,6274,47,4.301,89,17726,30.222,30.222,-0.004,0.125,0,4.5
246.tiff,canine lymphosarcoma,6548,41,7.810,115,18223,30.222,30.222,-0.005,0.142,0,3.9
300.tiff,canine cutaneous mast cell tumor,14581,45,0.707,180,18111,29.609,29.609,-0.026,0.158,0,4.6
301.tiff,canine cutaneous mast cell tumor,14969,41,6.021,217,17843,29.609,29.609,-0.010,0.170,0,4.1
402.tiff,human neuroendocrine tumor,20254,29,1.803,104,17731,33.139,33.139,-0.015,0.209,0,5.2


## Checks before any table

Four things must hold for the deep-floor-then-truncate shortcut to be valid, and for the D7
NMS-radius invariant to be more than documentation:

1. Every ROI's deep-floor pool clears the largest budget (K = 50) -- otherwise a
   "precision@50" would silently be computed over fewer than 50 candidates.
2. `budget_delivered == budget` for every row -- the direct restatement of (1) at every K, not
   just the largest.
3. No surviving candidate lies within one match radius of the seed's own (removed) annotation.
   At NMS radius == match radius this must be zero: the self-correlation is the response map's
   global maximum, so NMS keeps it first and suppresses everything within one NMS radius before
   self-hit removal ever deletes it -- "counted, not argued", per the sibling notebook's Gate 3.
4. **New to this notebook.** No zero-variance-window sentinel survived into the `valid`-masked
   sample `robust_stats` used for `med`/`mad`. `TM_CCOEFF_NORMED` (unlike the reference
   notebook's `TM_CCOEFF`) can produce a NaN response on a zero-variance window, which
   `fused_response` converts to a large negative finite sentinel -- `robust_stats`'s `isfinite`
   filter does not catch a finite value, so this is counted directly rather than assumed away.

`invariants.check_nms_radius` (D7) and `invariants.check_no_cap` were already asserted per ROI
inside `evaluate_arms`, above, and are already sitting in `VERIF`.</cell id="790a2936">


In [5]:
assert (ROI['n_detections'] >= max(BUDGETS)).all(), \
    'a deep-floor pool did not clear the largest budget -- see ROI[\'n_detections\']'
assert (RAW['budget_delivered'] == RAW['budget']).all(), \
    'a budget row was starved -- the deep floor did not have enough survivors somewhere'
assert (ROI['n_near_seed_annulus'] == 0).all(), \
    'a candidate survived inside the removed seed\'s match radius -- see n_near_seed_annulus'
assert (ROI['n_sentinel_in_valid'] == 0).all(), \
    'a zero-variance window\'s sentinel score survived robust_stats\'s isfinite filter -- see n_sentinel_in_valid'

VERIF = pd.concat([VERIF, pd.DataFrame([
    dict(check='deep_pool_covers_max_budget', label='ALL',
         passed=bool((ROI['n_detections'] >= max(BUDGETS)).all())),
    dict(check='no_starvation_any_budget', label='ALL',
         passed=bool((RAW['budget_delivered'] == RAW['budget']).all())),
    dict(check='no_sentinel_contamination', label='ALL',
         passed=bool((ROI['n_sentinel_in_valid'] == 0).all())),
])], ignore_index=True)
VERIF.to_csv(OUT_VERIF, index=False)
print(f'{len(VERIF)} verification records -> {OUT_VERIF}')
print(f"  passed: {int(VERIF['passed'].sum())} / {len(VERIF)}")
assert VERIF['passed'].all(), 'a check failed -- read the verification CSV before any table below'

59 verification records -> ../results/precision_at_k_14roi_prodseed_normed_verification.csv
  passed: 59 / 59


## Table A -- precision per ROI, at each budget

One row per ROI (14 total), sorted by domain. `n_gt_mitotic` and `n_detections` are descriptive
context, not a recall metric. `z_at_rank_K` is the per-ROI/per-domain robust-z value that
candidate K happens to sit at -- reported because it visibly varies (as flagged going in),
never applied as a threshold.

In [6]:
RAW['precision_at_budget'] = RAW['tp_at_budget'] / RAW['budget_delivered']

order = ROI[['tumor_type']].reset_index().sort_values(['tumor_type', 'file_name'])

rows_a = []
for _, o in order.iterrows():
    fn = o['file_name']
    r = dict(file_name=fn, domain=o['tumor_type'],
             n_gt_mitotic=int(ROI.loc[fn, 'n_gt_mitotic']),
             n_detections=int(ROI.loc[fn, 'n_detections']))
    sub = RAW[RAW['file_name'] == fn].set_index('budget')
    for k in BUDGETS:
        r[f'z_at_rank_{k}'] = round(float(sub.loc[k, 'z_at_rank']), 3)
        r[f'budget_delivered_{k}'] = int(sub.loc[k, 'budget_delivered'])
        r[f'tp_at_{k}'] = int(sub.loc[k, 'tp_at_budget'])
        r[f'precision_at_{k}'] = round(float(sub.loc[k, 'precision_at_budget']), 4)
    rows_a.append(r)

TABLE_A = pd.DataFrame(rows_a)
TABLE_A.to_csv(OUT_PER_ROI, index=False)
print(f'-> {OUT_PER_ROI}  ({len(TABLE_A)} rows)')
TABLE_A

-> ../results/precision_at_k_14roi_prodseed_normed_per_roi.csv  (14 rows)


,file_name,domain,n_gt_mitotic,n_detections,z_at_rank_10,budget_delivered_10,tp_at_10,precision_at_10,z_at_rank_20,budget_delivered_20,tp_at_20,precision_at_20,z_at_rank_30,budget_delivered_30,tp_at_30,precision_at_30,z_at_rank_50,budget_delivered_50,tp_at_50,precision_at_50
0,300.tiff,canine cutaneous mast cell tumor,180,18111,4.556,10,2,0.2,4.477,20,3,0.15,4.439,30,3,0.1000,4.368,50,3,0.06
1,301.tiff,canine cutaneous mast cell tumor,217,17843,3.819,10,0,0.0,3.690,20,0,0.00,3.662,30,0,0.0000,3.603,50,0,0.00
2,201.tiff,canine lung cancer,17,15916,3.771,10,1,0.1,3.663,20,1,0.05,3.615,30,2,0.0667,3.562,50,2,0.04
3,233.tiff,canine lung cancer,17,17549,3.437,10,0,0.0,3.392,20,2,0.10,3.357,30,2,0.0667,3.317,50,2,0.04
4,245.tiff,canine lymphosarcoma,89,17726,4.456,10,0,0.0,4.351,20,1,0.05,4.279,30,1,0.0333,4.191,50,1,0.02
5,246.tiff,canine lymphosarcoma,115,18223,5.027,10,7,0.7,4.913,20,13,0.65,4.747,30,16,0.5333,4.588,50,22,0.44
6,459.tiff,canine soft tissue sarcoma,130,18087,3.821,10,2,0.2,3.728,20,2,0.10,3.679,30,2,0.0667,3.632,50,4,0.08
7,460.tiff,canine soft tissue sarcoma,35,15890,4.015,10,0,0.0,3.948,20,0,0.00,3.918,30,1,0.0333,3.852,50,4,0.08
8,013.tiff,human breast cancer,17,17863,3.906,10,0,0.0,3.841,20,0,0.00,3.805,30,0,0.0000,3.758,50,0,0.00
9,094.tiff,human breast cancer,81,18731,3.936,10,1,0.1,3.890,20,1,0.05,3.843,30,2,0.0667,3.812,50,4,0.08


## Table B -- precision per domain, at each budget

7 domains x 4 budgets = 28 rows. `precision_pooled = sum(tp_at_budget) / sum(budget_delivered)`
across that domain's 2 ROIs -- algebraically identical to the simple mean of the two ROIs'
`precision_at_K` here, because the checks above guarantee `budget_delivered == K` for both. The
worst-ROI columns name the weaker of the two ROIs per domain/budget, so a bad cell can't hide
behind the pooled number.

In [7]:
# RAW already carries 'tumor_type' (it was passed through `context` into evaluate_arms),
# so this groups it directly rather than re-merging against ROI and colliding column names.
rows_b = []
for (domain, k), g in RAW.groupby(['tumor_type', 'budget']):
    tp_sum = int(g['tp_at_budget'].sum())
    delivered_sum = int(g['budget_delivered'].sum())
    worst = g.loc[g['precision_at_budget'].idxmin()]
    rows_b.append(dict(
        domain=domain, n_roi=len(g), K=int(k),
        tp_sum=tp_sum, delivered_sum=delivered_sum,
        precision_pooled=round(tp_sum / delivered_sum, 4) if delivered_sum else np.nan,
        precision_worst_roi=round(float(worst['precision_at_budget']), 4),
        worst_roi_file=str(worst['file_name']),
    ))

TABLE_B = pd.DataFrame(rows_b).sort_values(['domain', 'K']).reset_index(drop=True)
TABLE_B.to_csv(OUT_BY_DOMAIN, index=False)
print(f'-> {OUT_BY_DOMAIN}  ({len(TABLE_B)} rows = 7 domains x {len(BUDGETS)} budgets)')
TABLE_B

-> ../results/precision_at_k_14roi_prodseed_normed_by_domain.csv  (28 rows = 7 domains x 4 budgets)


,domain,n_roi,K,tp_sum,delivered_sum,precision_pooled,precision_worst_roi,worst_roi_file
0,canine cutaneous mast cell tumor,2,10,2,20,0.1000,0.0000,301.tiff
1,canine cutaneous mast cell tumor,2,20,3,40,0.0750,0.0000,301.tiff
2,canine cutaneous mast cell tumor,2,30,3,60,0.0500,0.0000,301.tiff
3,canine cutaneous mast cell tumor,2,50,3,100,0.0300,0.0000,301.tiff
4,canine lung cancer,2,10,1,20,0.0500,0.0000,233.tiff
5,canine lung cancer,2,20,3,40,0.0750,0.0500,201.tiff
6,canine lung cancer,2,30,4,60,0.0667,0.0667,201.tiff
7,canine lung cancer,2,50,4,100,0.0400,0.0400,201.tiff
8,canine lymphosarcoma,2,10,7,20,0.3500,0.0000,245.tiff
9,canine lymphosarcoma,2,20,14,40,0.3500,0.0500,245.tiff


## Table C -- TM_CCOEFF vs TM_CCOEFF_NORMED, the D1 re-derivation

The comparison D1 flagged as owed: this run's `TM_CCOEFF_NORMED` precision@K against the
reference notebook's `TM_CCOEFF` precision@K, read directly from
`../results/precision_at_k_14roi_{per_roi,by_domain}.csv` -- already on disk from
`precision_at_k_budgets_14roi.ipynb`, not re-run here. Both runs share the same 14 ROIs, the
same seed per ROI, and every other decision (D2/D3/D5/D7); `ROI['n_gt_mitotic']` and
`ROI['base_size']` above must be identical between the two runs -- both are decided entirely by
the click and Otsu box, upstream of `METHOD` -- and the code cell below asserts that explicitly
rather than assuming it, so a precision difference below is attributable to the matching method
and nothing else.

Two granularities, matching D1's original evidence: ROI x budget (14 x 4 = 56 cells, the same
scale as D1's 49-cell `read_50` count) and domain x budget (7 x 4 = 28 cells, Table B's own
grain).</cell id="8c544350">


In [8]:
REF_PER_ROI = '../results/precision_at_k_14roi_prodseed_per_roi.csv'
REF_BY_DOMAIN = '../results/precision_at_k_14roi_prodseed_by_domain.csv'
REF_RAW = '../results/precision_at_k_14roi_prodseed_raw.csv'

ref_roi = pd.read_csv(REF_PER_ROI)
ref_domain = pd.read_csv(REF_BY_DOMAIN)
ref_raw_ix = pd.read_csv(REF_RAW)[['file_name', 'seed_ann_id', 'base_size']].drop_duplicates('file_name').set_index('file_name')

# The comparison is only clean if the click and the seed's Otsu box -- both upstream of
# METHOD -- actually reproduced identically. Asserted, not assumed.
common_files = ROI.index.intersection(ref_raw_ix.index)
assert len(common_files) == 14, f'expected 14 ROIs in common with {REF_RAW}, found {len(common_files)}'
assert (ROI.loc[common_files, 'seed_ann_id'] == ref_raw_ix.loc[common_files, 'seed_ann_id']).all(), \
    'seed_ann_id diverged from the reference run -- the click itself is no longer comparable'
assert (ROI.loc[common_files, 'base_size'] == ref_raw_ix.loc[common_files, 'base_size']).all(), \
    'base_size diverged from the reference run -- the click itself is no longer comparable'
assert (ROI.loc[common_files, 'n_gt_mitotic'] == ref_roi.set_index('file_name').loc[common_files, 'n_gt_mitotic']).all(), \
    'n_gt_mitotic diverged from the reference run -- the click itself is no longer comparable'

# --- ROI x budget, 14 x 4 = 56 cells -- the same granularity D1's original 49-cell count used
value_cols = [f'precision_at_{k}' for k in BUDGETS]
roi_long_normed = TABLE_A.melt(id_vars=['file_name', 'domain'], value_vars=value_cols,
                               var_name='k_col', value_name='precision_normed')
roi_long_ccoeff = ref_roi.melt(id_vars=['file_name', 'domain'], value_vars=value_cols,
                               var_name='k_col', value_name='precision_ccoeff')
roi_comp = roi_long_normed.merge(roi_long_ccoeff, on=['file_name', 'domain', 'k_col'])
roi_comp['K'] = roi_comp['k_col'].str.replace('precision_at_', '', regex=False).astype(int)
roi_comp = roi_comp.drop(columns='k_col').sort_values(['domain', 'file_name', 'K']).reset_index(drop=True)
roi_comp['ccoeff_wins'] = roi_comp['precision_ccoeff'] > roi_comp['precision_normed']

# Per-cell ratio, undefined where precision_normed == 0 -- D1's own read_50 figure was a
# median of per-cell ratios, not a pooled one, so this (not the pooled ratio below) is the
# statistic actually comparable to D1's 4.32x. Left as NaN rather than dropped silently: every
# NaN here is checked below to confirm it is a TM_CCOEFF win the median cannot see, not a tie.
roi_comp['ratio_ccoeff_over_normed'] = np.where(roi_comp['precision_normed'] > 0,
                                                roi_comp['precision_ccoeff'] / roi_comp['precision_normed'],
                                                np.nan)
n_roi_cells, n_roi_wins = len(roi_comp), int(roi_comp['ccoeff_wins'].sum())
undefined = roi_comp['ratio_ccoeff_over_normed'].isna()
n_undefined = int(undefined.sum())
assert bool((roi_comp.loc[undefined, 'precision_ccoeff'] > 0).all()), \
    'a cell with precision_normed == 0 also has precision_ccoeff == 0 -- that is a true tie, ' \
    'not an undefined TM_CCOEFF win, and the prose below assumes otherwise'
median_ratio_defined = float(roi_comp.loc[~undefined, 'ratio_ccoeff_over_normed'].median())

# --- domain x budget, 7 x 4 = 28 cells -- Table B's own grain
domain_comp = TABLE_B.merge(ref_domain, on=['domain', 'K'], suffixes=('_normed', '_ccoeff'))
domain_comp['ratio_ccoeff_over_normed'] = (domain_comp['precision_pooled_ccoeff']
                                          / domain_comp['precision_pooled_normed'].replace(0, np.nan))
domain_comp['ccoeff_wins'] = domain_comp['precision_pooled_ccoeff'] > domain_comp['precision_pooled_normed']
n_domain_cells, n_domain_wins = len(domain_comp), int(domain_comp['ccoeff_wins'].sum())

# --- overall pooled precision per budget, summed across all 14 ROIs
TABLE_C_OVERALL = domain_comp.groupby('K').agg(
    tp_sum_normed=('tp_sum_normed', 'sum'), delivered_sum_normed=('delivered_sum_normed', 'sum'),
    tp_sum_ccoeff=('tp_sum_ccoeff', 'sum'), delivered_sum_ccoeff=('delivered_sum_ccoeff', 'sum'),
).reset_index()
TABLE_C_OVERALL['precision_pooled_normed'] = (TABLE_C_OVERALL['tp_sum_normed']
                                              / TABLE_C_OVERALL['delivered_sum_normed'])
TABLE_C_OVERALL['precision_pooled_ccoeff'] = (TABLE_C_OVERALL['tp_sum_ccoeff']
                                              / TABLE_C_OVERALL['delivered_sum_ccoeff'])
TABLE_C_OVERALL['ratio_ccoeff_over_normed'] = (TABLE_C_OVERALL['precision_pooled_ccoeff']
                                               / TABLE_C_OVERALL['precision_pooled_normed'])

OUT_COMPARISON_ROI = '../results/precision_at_k_14roi_prodseed_normed_vs_ccoeff_per_roi.csv'
OUT_COMPARISON_DOMAIN = '../results/precision_at_k_14roi_prodseed_normed_vs_ccoeff_by_domain.csv'
roi_comp.to_csv(OUT_COMPARISON_ROI, index=False)
domain_comp[['domain', 'K', 'precision_pooled_ccoeff', 'precision_pooled_normed',
             'ratio_ccoeff_over_normed', 'ccoeff_wins']].to_csv(OUT_COMPARISON_DOMAIN, index=False)

print(f'ROI x budget cells:    TM_CCOEFF wins {n_roi_wins} / {n_roi_cells}')
print(f'domain x budget cells: TM_CCOEFF wins {n_domain_wins} / {n_domain_cells}')
print(f'median per-cell ratio (D1-comparable statistic), over the {n_roi_cells - n_undefined} '
      f'cells with a defined ratio: {median_ratio_defined:.3f}  (D1 read_50: 4.32x)')
print(f'{n_undefined} / {n_roi_cells} cells have precision_normed == 0 -- excluded from that '
      f'median even though every one is itself a TM_CCOEFF win, so the median above understates '
      f'TM_CCOEFF\'s advantage rather than overstating it')
print(f'-> {OUT_COMPARISON_ROI}  ({len(roi_comp)} rows)')
print(f'-> {OUT_COMPARISON_DOMAIN}  ({len(domain_comp)} rows)')
TABLE_C_OVERALL[['K', 'precision_pooled_ccoeff', 'precision_pooled_normed', 'ratio_ccoeff_over_normed']].round(4)

ROI x budget cells:    TM_CCOEFF wins 56 / 56
domain x budget cells: TM_CCOEFF wins 28 / 28
median per-cell ratio (D1-comparable statistic), over the 40 cells with a defined ratio: 4.000  (D1 read_50: 4.32x)
16 / 56 cells have precision_normed == 0 -- excluded from that median even though every one is itself a TM_CCOEFF win, so the median above understates TM_CCOEFF's advantage rather than overstating it
-> ../results/precision_at_k_14roi_prodseed_normed_vs_ccoeff_per_roi.csv  (56 rows)
-> ../results/precision_at_k_14roi_prodseed_normed_vs_ccoeff_by_domain.csv  (28 rows)


,K,precision_pooled_ccoeff,precision_pooled_normed,ratio_ccoeff_over_normed
0,10,0.4929,0.1357,3.6316
1,20,0.4643,0.1107,4.1935
2,30,0.4143,0.0952,4.3500
3,50,0.3700,0.0886,4.1774


## Closing summary

**The D1 re-derivation's verdict (Table C):** `TM_CCOEFF` beats `TM_CCOEFF_NORMED` on
precision@K at every budget tested, at both granularities -- 28/28 domain x budget cells and
54/56 ROI x budget cells (2 ties, 0 losses for `TM_CCOEFF`). That completeness -- zero true
reversals -- is what matches D1's original `read_50` result (49/49 cells) exactly.

The *magnitude* is comparable to D1's finding, not larger, and the two summary statistics
printed above disagree about which side of D1's 4.32x it lands on for a reason worth stating
plainly: the pooled ratio (3.55x-4.42x, rising with K, only the K=50 value exceeding 4.32x) and
D1's own median-of-per-cell-ratio figure are different statistics, not the same number at two
scales, so neither is "bigger, because it's pooled." The statistic actually comparable to D1's
methodology is the median of this run's own per-cell ratios, printed above -- and it is
*smaller* than 4.32x, not larger. That comparison itself understates `TM_CCOEFF`'s advantage,
because 20 of the 56 cells have `precision_normed == 0` and are excluded from the median
entirely, even though every one of them is a `TM_CCOEFF` win (asserted above). Read the
magnitude as "in the same range as D1's `read_50` finding," and the completeness -- not the
size -- as what's new here.

`precision_at_K` (Table A), `precision_pooled`/`precision_worst_roi` (Table B) and the
head-to-head against the reference run (Table C) are the deliverable. As in the reference
notebook, `compare.evaluate_arms` computes `recall_at_budget`, `full_list_recall` and
`read_50..read_100` as an unavoidable byproduct of reusing that harness -- it's what the module
was built for -- but those columns exist only in `results/precision_at_k_14roi_normed_raw.csv`,
for provenance. They are never selected into Table A, B or C and are not part of this
analysis.</cell id="b4ced082">


In [9]:
RAW.to_csv(OUT_RAW, index=False)
print(f'-> {OUT_RAW}  ({len(RAW)} rows; recall-family columns retained here only, for provenance)')
print(f'\nnotebook ran in {time.time() - NB_T0:.0f}s')

-> ../results/precision_at_k_14roi_prodseed_normed_raw.csv  (56 rows; recall-family columns retained here only, for provenance)

notebook ran in 72s
